# Análisis de datos

In [71]:
import altair as alt

## Código sacado de offline_eval.ipynb

In [ ]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q) -> pd.DataFrame:
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

Ejemplo:

In [2]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0383


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_ee27a18380ed096b,2026-11-10 03:05:49.533000000,u_782a47c0,"{'device': 'android', 'page': 'product', 'regi...","[{'product_id': 'p_8914eb', 'ts': '2026-11-10 ...","[p_208432, p_adc3e6, p_f391d6, p_08b1a0, p_09a...","[0.001163, 0.268949, 0.000237, 0.008724, 0.156...",NaN,NaN
1,r_eefcd62e62a73c07,2026-11-13 03:34:26.883000000,u_95384ed4,"{'device': 'ios', 'page': 'product', 'region':...","[{'product_id': 'p_93aeec', 'ts': '2026-11-13 ...","[p_82afd0, p_5639e2, p_fcf075, p_65b5a3, p_664...","[0.003962, 0.029644, 0.003122, 0.004293, 0.002...",p_52e7e7,p_52e7e7
2,r_7fc058a4fedc2182,2026-11-13 03:54:42.765000000,u_c83e4f46,"{'device': 'android', 'page': 'category', 'reg...","[{'product_id': 'p_370ec7', 'ts': '2026-11-13 ...","[p_5733ef, p_9f6f74, p_7ca398, p_bc0984, p_2f0...","[0.368928, 0.016695, 0.089816, 0.029263, 0.179...",NaN,NaN


## Más datos

Nota: podemos ver todas las tablas porque hemos empezado a hacer submissions con el baseline (guía en el README).

### Consulta equivalente a .head() de pandas en las diferentes tablas

In [9]:
def query_head(tabla: str, n: int = 5) -> pd.DataFrame:
    """Equivalente a pd.head con SQL."""
    query = f"SELECT * from {tabla} LIMIT {n}"
    return sql(query)

#### Available from the start

In [11]:
query_head("shop.catalog")

,brand,category,created_at,description,is_active,price_eur,product_id,subcategory
0,Vedo,Sports & Outdoors,2026-01-21 07:00:00.000000000,Vedo Yoga Mats with organic ingredients and ex...,True,14.99,p_001897,Yoga Mats
1,Zore,Fashion,2025-11-06 07:00:00.000000000,"From Zore (Jackets): wireless connectivity, a ...",True,4.99,p_00196b,Jackets
2,Sanine,Electronics,2025-01-28 07:00:00.000000000,Sanine Laptop Sleeves with a luxury feel and a...,True,57.99,p_001a20,Laptop Sleeves
3,Vebexa,Sports & Outdoors,2024-09-18 07:00:00.000000000,Vebexa Running Shoes with built for profession...,True,29.99,p_001d7f,Running Shoes
4,Nibe,Home & Kitchen,2025-10-15 07:00:00.000000000,Nibe Cutting Boards with a minimalist design a...,True,26.99,p_0032c5,Cutting Boards


In [ ]:
query_head("shop.history_sessions")

,clicked,context.device,context.page,context.region,propensity,purchased,recent_views,request_id,revenue_eur,shown,ts,user_id
0,None,web,product,Catalunya,"[0.218886, 0.002833, 0.11516, 0.321647, 0.0292...",None,"[{'product_id': 'p_159190', 'ts': '2026-09-15 ...",r_50309edc92778291,0,"[p_8ca8ed, p_8ac75a, p_e2c959, p_2c88d5, p_8fb...",2026-09-15 07:00:01.456000000,u_20cbc5d5
1,None,web,product,Madrid,"[0.032807, 0.011809, 0.008224, 0.025852, 0.290...",None,"[{'product_id': 'p_f3c09e', 'ts': '2026-09-15 ...",r_7a441a0fb217005e,0,"[p_032d24, p_40bbb4, p_66f831, p_d245de, p_adc...",2026-09-15 07:00:13.975000000,u_cd609f53
2,None,android,category,Aragón,"[0.009835, 0.271305, 0.005089, 0.050632, 0.165...",None,"[{'product_id': 'p_f41d0b', 'ts': '2026-09-15 ...",r_7fa58169a04e436c,0,"[p_9c2b8b, p_adc3e6, p_883ab9, p_07b81b, p_09a...",2026-09-15 07:00:19.633000000,u_5a69bbf5
3,None,ios,product,Aragón,"[0.003099, 0.219566, 0.019525, 0.290605, 0.007...",None,"[{'product_id': 'p_07bab2', 'ts': '2026-09-15 ...",r_28800da566ece94b,0,"[p_fe5a63, p_8ca8ed, p_b8227f, p_2c88d5, p_819...",2026-09-15 07:00:22.418000000,u_039914dd
4,None,android,product,Catalunya,"[0.004059, 0.219778, 0.285283, 0.16151, 0.0326...",None,"[{'product_id': 'p_f67b9c', 'ts': '2026-09-15 ...",r_a3372ac6405c3511,0,"[p_ef8559, p_8ca8ed, p_2c88d5, p_e2c959, p_4ba...",2026-09-15 07:00:47.069000000,u_79d89af1


In [12]:
query_head("shop.history_events")

,device,event_type,price_eur,product_id,region,session_id,ts,user_id
0,android,view,9.99,p_d06b48,Catalunya,s_9aee4d52a08a,2026-09-15 07:00:06.371000000,u_04b42c08
1,web,view,85.99,p_9dc613,Comunitat Valenciana,s_8e7292024915,2026-09-15 07:00:13.592000000,u_9f64d903
2,android,view,215.99,p_134914,Catalunya,s_9aee4d52a08a,2026-09-15 07:00:23.341000000,u_04b42c08
3,web,view,9.99,p_f8920d,Catalunya,s_92b57e4008c4,2026-09-15 07:00:45.091000000,u_ffc15e95
4,web,view,30.99,p_f709f7,Comunitat Valenciana,s_8e7292024915,2026-09-15 07:01:01.182000000,u_9f64d903


#### During the event (Saturday to Sunday)

In [14]:
query_head("shop.requests")

,context.device,context.page,context.region,is_new_user,recent_views,request_id,ts,user_id
0,ios,product,Castilla y León,False,"[{'product_id': 'p_354478', 'ts': '2026-11-14 ...",r_3490d43fc4edca27,2026-11-14 09:00:00.014000000,u_dde95cc2
1,android,product,Murcia,True,"[{'product_id': 'p_5be836', 'ts': '2026-11-14 ...",r_be9a6b921540ee06,2026-11-14 09:00:00.382000000,u_157775db
2,android,home,Andalucía,True,[],r_5a80914f0566f217,2026-11-14 09:00:00.687000000,u_41216ce0
3,ios,product,Catalunya,True,"[{'product_id': 'p_170f13', 'ts': '2026-11-14 ...",r_a9ea6a9e25eb3f6a,2026-11-14 09:00:01.041000000,u_061f799b
4,ios,product,País Vasco,True,"[{'product_id': 'p_32773e', 'ts': '2026-11-14 ...",r_5c385016b1d75ace,2026-11-14 09:00:01.352000000,u_f2fcbfa3


Básicamente tenemos que estar mirando esta tabla constantemente para ver si han llegado nuevos clientes.

In [15]:
query_head("shop.events")

,device,event_type,price_eur,product_id,region,session_id,ts,user_id
0,ios,view,25.99,p_931b47,Andalucía,s_d15161d3da9d,2026-11-14 09:00:00.014000000,u_a7c85306
1,ios,view,39.99,p_0468c1,Andalucía,s_d15161d3da9d,2026-11-14 09:00:00.021000000,u_a7c85306
2,ios,view,37.99,p_0cb387,Andalucía,s_d15161d3da9d,2026-11-14 09:00:00.028000000,u_a7c85306
3,ios,view,25.99,p_164481,Andalucía,s_d15161d3da9d,2026-11-14 09:00:00.035000000,u_a7c85306
4,web,view,21.99,p_d72d81,Andalucía,s_463986306ad7,2026-11-14 09:00:00.085000000,u_c13fe3d2


In [16]:
query_head("shop.catalog_updates")

,brand,category,change,created_at,description,is_active,price_eur,product_id,subcategory,ts
0,Maka,Sports & Outdoors,out_of_stock,2024-11-05 07:00:00.000000000,Maka Running Shoes with a vintage look and com...,False,11.99,p_66211c,Running Shoes,2026-11-14 09:00:00.000000000
1,Tinove,Sports & Outdoors,out_of_stock,2026-09-12 07:00:00.000000000,Tinove Bicycles with made from recycled materi...,False,48.99,p_8c4c8d,Bicycles,2026-11-14 09:00:00.000000000
2,Minoma,Books & Media,out_of_stock,2025-04-10 07:00:00.000000000,"E-Reader Covers by Minoma, featuring a vintage...",False,256.99,p_5f97a9,E-Reader Covers,2026-11-14 09:00:00.000000000
3,Maka,Sports & Outdoors,out_of_stock,2026-08-19 07:00:00.000000000,"Maka Tents line, known for built for professio...",False,44.99,p_f0bc6c,Tents,2026-11-14 09:05:00.000000000
4,Kofi,Electronics,out_of_stock,2025-07-02 07:00:00.000000000,"From Kofi (Laptop Sleeves): bold colours, a mi...",False,34.99,p_eab680,Laptop Sleeves,2026-11-14 09:05:00.000000000


In [17]:
query_head("team_30.feedback")

,clicked,fallback_reason,latency_ms,purchased,request_id,revenue_eur,served_by,shown,ts
0,,,0.0,,__init__,0,init,[],2026-01-01 00:00:00.000000000
1,,missing,NaN,,r_39adfc7374a28fa8,0,fallback,"[p_0210c5, p_09a938, p_1036d3, p_e2454f, p_2c8...",2026-11-14 11:20:22.247000000
2,,missing,NaN,,r_267e6a225ef7debd,0,fallback,"[p_0210c5, p_09a938, p_1036d3, p_e2454f, p_2c8...",2026-11-14 11:20:22.602000000
3,,missing,NaN,,r_a5d34246899ca29d,0,fallback,"[p_0210c5, p_09a938, p_1036d3, p_e2454f, p_2c8...",2026-11-14 11:20:22.921000000
4,,missing,NaN,,r_c1eead79cc568451,0,fallback,"[p_0210c5, p_09a938, p_1036d3, p_e2454f, p_2c8...",2026-11-14 11:20:23.236000000


Nota: nos interesa ordenar por fecha para tener las últimas.

In [19]:
sql("SELECT * from team_30.feedback ORDER BY ts DESC LIMIT 5")

,clicked,fallback_reason,latency_ms,purchased,request_id,revenue_eur,served_by,shown,ts
0,,,1256,,r_0703f5b3a359e9fe,0.00,team,"[p_adc3e6, p_69d40e, p_0210c5, p_1036d3, p_e24...",2026-11-14 12:12:32.589000000
1,,,1573,,r_924bd9c05a757f87,0.00,team,"[p_adc3e6, p_69d40e, p_0210c5, p_1036d3, p_e24...",2026-11-14 12:12:32.233000000
2,,,1912,,r_3cbdd0d122e89222,0.00,team,"[p_adc3e6, p_69d40e, p_0210c5, p_1036d3, p_e24...",2026-11-14 12:12:31.916000000
3,p_1036d3,,2218,p_1036d3,r_dd9f01b2a7eb2417,13.99,team,"[p_adc3e6, p_69d40e, p_0210c5, p_1036d3, p_e24...",2026-11-14 12:12:31.628000000
4,,,2586,,r_ed0842f7bfd9e0f3,0.00,team,"[p_adc3e6, p_69d40e, p_0210c5, p_1036d3, p_e24...",2026-11-14 12:12:31.222000000


We see that now "served_by" is "team", which means the server has received proper answers.

## Análisis

### Catálogo

El catálogo tiene unos 5000 productos, en unas 10 categorías y 90 subcategorías.

- ¿Cómo se distribuyen esas categorías y sbucategorías?

No se da información de las marcas.

Cada producto tiene un precio.

- Alerta: el precio cambia en shop.catalog_updates; el de esta tabla es inicial.

- Alerta: también puede ser que un producto pase a estar activo o deje de estarlo.

Podemos suponer que el precio, la marca, categoría y subcategoría serán decisivas.

- ¿Es posible que la descripción también sea relevante? Al ser texto, sería interesante sacar variables latentes que estén aquí.

In [21]:
catalog_df_sample = query_head("shop.catalog", 50)
catalog_df_sample.head()

,brand,category,created_at,description,is_active,price_eur,product_id,subcategory
0,Vedo,Sports & Outdoors,2026-01-21 07:00:00.000000000,Vedo Yoga Mats with organic ingredients and ex...,True,14.99,p_001897,Yoga Mats
1,Zore,Fashion,2025-11-06 07:00:00.000000000,"From Zore (Jackets): wireless connectivity, a ...",True,4.99,p_00196b,Jackets
2,Sanine,Electronics,2025-01-28 07:00:00.000000000,Sanine Laptop Sleeves with a luxury feel and a...,True,57.99,p_001a20,Laptop Sleeves
3,Vebexa,Sports & Outdoors,2024-09-18 07:00:00.000000000,Vebexa Running Shoes with built for profession...,True,29.99,p_001d7f,Running Shoes
4,Nibe,Home & Kitchen,2025-10-15 07:00:00.000000000,Nibe Cutting Boards with a minimalist design a...,True,26.99,p_0032c5,Cutting Boards


In [32]:
descriprion_samples = list(catalog_df_sample["description"][:10])
descriprion_samples

['Vedo Yoga Mats with organic ingredients and extra durability.',
 'From Zore (Jackets): wireless connectivity, a travel-ready build.',
 'Sanine Laptop Sleeves with a luxury feel and a waterproof shell.',
 'Vebexa Running Shoes with built for professionals and a luxury feel.',
 'Nibe Cutting Boards with a minimalist design and premium materials.',
 'Nika Vitamins with a waterproof shell and a handmade finish.',
 'Zofiri Socks line, known for great value for money and bold colours.',
 'Rixa Dog Toys with a minimalist design and a sporty style.',
 'Strollers by Tixa, featuring wireless connectivity. Comes with organic ingredients.',
 'From Luzo (Coffee Machines): compact and lightweight, premium materials.']

- ¿Se podrían extraer nuevas variables, como alguna que defina el nivel de lujo, o que acompañen a las cateogrías?

In [31]:
catalog_df_sample.head(5)

,brand,category,created_at,description,is_active,price_eur,product_id,subcategory
0,Vedo,Sports & Outdoors,2026-01-21 07:00:00.000000000,Vedo Yoga Mats with organic ingredients and ex...,True,14.99,p_001897,Yoga Mats
1,Zore,Fashion,2025-11-06 07:00:00.000000000,"From Zore (Jackets): wireless connectivity, a ...",True,4.99,p_00196b,Jackets
2,Sanine,Electronics,2025-01-28 07:00:00.000000000,Sanine Laptop Sleeves with a luxury feel and a...,True,57.99,p_001a20,Laptop Sleeves
3,Vebexa,Sports & Outdoors,2024-09-18 07:00:00.000000000,Vebexa Running Shoes with built for profession...,True,29.99,p_001d7f,Running Shoes
4,Nibe,Home & Kitchen,2025-10-15 07:00:00.000000000,Nibe Cutting Boards with a minimalist design a...,True,26.99,p_0032c5,Cutting Boards


Está claro que tanto los productos con id p_001897 y p_001d7f son "Sports & Outdoors", pero su precio y descripción tienen target clients totalmente diferentes.

La descripción codifica:
- Calidad ("premium", "luxury"; "value for money")
- Nichos, preferencias personales... ("organic", "minimalist", "handmade")

Solamente en las primeras diez vemos patrones repetidos:
- "minimalist design"
- "premium materials"
- "organic ingredients"
- "wireless connectivity"
- "waterproof shell"

Una posibilidad es buscar todas las parejas de palabras que se repitan en las descripciones más comunes y codificarlas como booleanos, obteniendo uno por producto.

Utilizar la descripción por si misma, en crudo, no es factible si no se usa un modelo de lenguaje (al menos, no es práctico y no se espera buen resultado). Por tanto, se reemplazaría la descripción por este conjunto de booleanos, a los que llamaremos **tags** de producto.

Evidentemente, si analizamos las descripciones, esto se debe precomputar (computarlo al inicio para todas, o computarlo a la mínima que se añada un producto al catálogo).

- Esta función de añadir un catálogo al índice y procesarlo es muy importante, porque shop.catalog_updates puede añadir productos nuevos al catálogo.

- Posible fallback: si no hemos etiquetado un producto, establecer etiquetas de alguna manera (por ejemplo random o etiqueta "genérica". El problema de una nueva etiqueta es que haya cero ejemplares en el entrenamiento).

#### Cuestión: ¿todo producto en shop.catalog_updates estará en shop.catalog? **NO**

In [36]:
new_products_df = sql("""
SELECT count(DISTINCT product_id) AS n_new_products
FROM (
    SELECT product_id::String AS product_id
    FROM shop.catalog_updates
)
WHERE product_id NOT IN (
    SELECT product_id::String FROM shop.catalog
);
""")
new_products_df

,n_new_products
0,100


In [39]:
catalog_ids = sql("""
SELECT product_id::String AS product_id
FROM shop.catalog
GROUP BY product_id
""")

updates_df = sql("""
SELECT product_id::String AS product_id,
       count()            AS n_events,
       min(ts)            AS first_seen,
       max(ts)            AS last_seen
FROM shop.catalog_updates
GROUP BY product_id
""")

new_df = updates_df[~updates_df["product_id"].isin(catalog_ids["product_id"])]
print(len(new_df))                      # debería salir 100
new_df.sort_values("first_seen").head(10)

100


,product_id,n_events,first_seen,last_seen
7,p_fc0cd6,1,2026-11-14 10:00:00.000000000,2026-11-14 10:00:00.000000000
624,p_cffe83,1,2026-11-14 10:00:00.000000000,2026-11-14 10:00:00.000000000
622,p_d6f719,1,2026-11-14 10:00:00.000000000,2026-11-14 10:00:00.000000000
617,p_d44504,1,2026-11-14 10:00:00.000000000,2026-11-14 10:00:00.000000000
609,p_89b937,1,2026-11-14 10:00:00.000000000,2026-11-14 10:00:00.000000000
590,p_10c90f,1,2026-11-14 10:00:00.000000000,2026-11-14 10:00:00.000000000
549,p_469b63,1,2026-11-14 10:00:00.000000000,2026-11-14 10:00:00.000000000
532,p_7ab07e,1,2026-11-14 10:00:00.000000000,2026-11-14 10:00:00.000000000
519,p_748772,1,2026-11-14 10:00:00.000000000,2026-11-14 10:00:00.000000000
517,p_182efd,1,2026-11-14 10:00:00.000000000,2026-11-14 10:00:00.000000000


Se confirma que catalog.shop no tiene todos los productos. Por lo tanto, se debe indexar a medida que se crean productos nuevos.

¿Cuántas veces se han añadido productos?

In [40]:
new_df["first_seen"].unique()

<StringArray>
['2026-11-14 10:00:00.000000000']
Length: 1, dtype: str

Interesante, todos se añaden de una tacada.

In [44]:
sql("SELECT * FROM shop.catalog WHERE product_id='p_fc0cd6'")

""


In [45]:
sql("SELECT * FROM shop.catalog_updates WHERE product_id='p_fc0cd6'")

,brand,category,change,created_at,description,is_active,price_eur,product_id,subcategory,ts
0,Zota,Home & Kitchen,new_product,2026-11-14 10:00:00.000000000,Zota Toasters with wireless connectivity and g...,True,16.99,p_fc0cd6,Toasters,2026-11-14 10:00:00.000000000


Más claro el agua.

#### Consultas premade

In [46]:
current_catalogue_df = sql("""
SELECT product_id,
       argMax(category, v)    AS category,
       argMax(subcategory, v) AS subcategory,
       argMax(price_eur, v)   AS price_eur,
       argMax(is_active, v)   AS is_active
FROM (
    SELECT product_id::String AS product_id, category::String AS category, subcategory::String AS subcategory,
           price_eur::Float64 AS price_eur, is_active::Bool AS is_active, '' AS v
    FROM shop.catalog
    UNION ALL
    SELECT product_id::String, category::String, subcategory::String, price_eur::Float64, is_active::Bool, toString(ts)
    FROM shop.catalog_updates
)
GROUP BY product_id;
""")
current_catalogue_df.head()

,product_id,category,subcategory,price_eur,is_active
0,p_6d134f,Toys & Baby,Toy Batteries,33.99,True
1,p_435542,Home & Kitchen,Toasters,45.99,True
2,p_be1b74,Toys & Baby,Baby Bottles,13.99,True
3,p_7d7c66,Pets,Cat Litter,24.99,True
4,p_cf595e,Beauty & Health,Toothbrushes,11.99,True


In [47]:
popularity_per_category_last_h_df = sql("""
SELECT c.category::String AS category, e.product_id::String AS product_id,
       countIf(e.event_type = 'view')     AS views,
       countIf(e.event_type = 'purchase') AS purchases
FROM shop.events AS e
INNER JOIN shop.catalog AS c ON c.product_id::String = e.product_id::String
WHERE e.ts > now() - INTERVAL 1 HOUR
GROUP BY category, product_id
ORDER BY purchases DESC, views DESC
LIMIT 5 BY category;
""")
popularity_per_category_last_h_df.head()

,category,product_id,views,purchases
0,Books & Media,p_adc3e6,399,131
1,Computers & Gaming,p_e2454f,303,86
2,Toys & Baby,p_0210c5,284,77
3,Toys & Baby,p_1036d3,257,71
4,Electronics,p_2c88d5,253,64


In [48]:
subcategory_trend_comparison_df = sql("""
SELECT c.subcategory::String AS subcategory,
       countIf(e.ts > now() - INTERVAL 15 MINUTE)        AS views_15m,
       countIf(e.ts <= now() - INTERVAL 15 MINUTE) / 12  AS avg_views_per_15m_before,
       views_15m / greatest(avg_views_per_15m_before, 1) AS lift
FROM shop.events AS e
INNER JOIN shop.catalog AS c ON c.product_id::String = e.product_id::String
WHERE e.event_type = 'view' AND e.ts > now() - INTERVAL 195 MINUTE
GROUP BY subcategory
ORDER BY lift DESC
LIMIT 10;
""")
subcategory_trend_comparison_df.head()

,subcategory,views_15m,avg_views_per_15m_before,lift
0,Sofas,6,3.583333,1.674419
1,Fitness Trackers,614,471.416667,1.302457
2,Sunscreen,282,230.250000,1.224756
3,E-Readers,1123,924.000000,1.215368
4,Socks,781,655.000000,1.192366


In [53]:
coviews_df = sql("""
SELECT b.product_id AS also_viewed, count() AS sessions
FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view') AS a
INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view') AS b
    ON a.session_id = b.session_id
WHERE a.product_id = 'p_xxxxxx' AND b.product_id != a.product_id
GROUP BY also_viewed
ORDER BY sessions DESC
LIMIT 20;
""")
print(coviews_df.info())
coviews_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame
None


""


##### Feedback y reliability dan error

In [55]:
feedback_df = sql("""
SELECT toStartOfHour(ts::DateTime64(3)) AS hour,
       served_by::String                   AS served_by,
       count()                             AS requests,
       100 * countIf(clicked != '') / count()   AS ctr_pct,
       100 * countIf(purchased != '') / count() AS conversion_pct,
       sum(revenue_eur::Float64) / count()      AS revenue_per_request
FROM team_NN.feedback
WHERE request_id != '__init__'
GROUP BY hour, served_by
ORDER BY hour DESC, served_by;
""")
feedback_df.head()

HTTPError: 400 Client Error: Bad Request for url: https://api.rawtree.com/v1/query

In [56]:
reliability_df = sql("""
SELECT fallback_reason::String              AS fallback_reason,
       count()                              AS requests,
       quantile(0.5)(latency_ms::Float64)   AS p50_ms,
       quantile(0.95)(latency_ms::Float64)  AS p95_ms
FROM team_NN.feedback
WHERE ts > now() - INTERVAL 30 MINUTE AND request_id != '__init__'
GROUP BY fallback_reason;


-- 7. Which slots get clicked? (position bias in the old system's log)
SELECT indexOf(shown, clicked) AS slot, count() AS clicks
FROM shop.history_sessions
WHERE clicked IS NOT NULL
GROUP BY slot
ORDER BY slot;
""")
reliability_df.head()

HTTPError: 400 Client Error: Bad Request for url: https://api.rawtree.com/v1/query

#### Frecuencias

- ¿Cada cuánto aparece un nuevo usuario?

In [ ]:
if False:
    # Todos los eventos
    events_df = sql("SELECT ts FROM shop.events ORDER BY ts")
    events_df.head(5)

if True:
    # Eventos entre 2 momentos
    START = '2026-11-14 13:00:00'
    END   = '2026-11-14 14:00:00'

    events_df = sql(f"""
        SELECT ts
        FROM shop.events
        WHERE ts >= '{START}'
        AND ts <= '{END}'
        ORDER BY ts
    """)

    print(len(events_df), "events")
    events_df.head()

239957 events


In [93]:
df = events_df.copy()

# Convert to datetime
df['ts'] = pd.to_datetime(df['ts'])

# Calculate intervals in seconds
intervals = df['ts'].diff().dt.total_seconds()
intervals = intervals.dropna()
intervals = intervals[intervals >= 0]

# Define bin width in seconds
bin_width = 0.0005 

# Aggregate into numeric bins
bin_start = (intervals // bin_width) * bin_width

histogram = (
    bin_start.value_counts()
    .sort_index()
    .rename_axis('bin_start')
    .reset_index(name='count')
)

histogram['bin_end'] = histogram['bin_start'] + bin_width
histogram['bin_end'] = histogram['bin_start'] + bin_width

# Altair chart
chart = alt.Chart(histogram).mark_bar().encode(
    x=alt.X(
        'bin_start:Q',
        title='Time interval (seconds)',
        axis=alt.Axis(tickMinStep=bin_width)
    ),
    x2='bin_end:Q',
    y=alt.Y('count:Q', title='Number of occurrences'),
    tooltip=[
        alt.Tooltip('bin_start:Q', title='From (seconds)'),
        alt.Tooltip('bin_end:Q', title='To (seconds)'),
        alt.Tooltip('count:Q', title='Occurrences')
    ]
).properties(
    title='Distribution of intervals between consecutive events',
    width=750,
    height=400
)

chart

alt.Chart(...)

Parece que lo normal es que haya 0,007 segundos entre eventos. Eso son unas 150 por segundo, no 3. Los 3 por segundo se refiere, por lo tanto, a otra cosa.

Quizás cambia si forzamos que los clientes sean diferentes.

In [94]:
# Eventos entre 2 momentos
START = '2026-11-14 13:00:00'
END   = '2026-11-14 14:00:00'

events_df = sql(f"""
    SELECT ts, user_id
    FROM shop.events
    WHERE ts >= '{START}'
        AND ts <= '{END}'
    ORDER BY ts
""")

print(len(events_df), "events")
events_df.head()

df = events_df.copy()

# Convert to datetime
df['ts'] = pd.to_datetime(df['ts'])

# Sort chronologically
df = df.sort_values('ts').reset_index(drop=True)

# Calculate intervals between consecutive events
df['interval'] = df['ts'].diff().dt.total_seconds()

# Keep only intervals where consecutive events belong to different users
different_user = df['user_id'].ne(df['user_id'].shift())

intervals = df.loc[different_user, 'interval'].dropna()

# Remove negative intervals
intervals = intervals[intervals >= 0]

# Define bin width in seconds
bin_width = 0.0005

# Aggregate into numeric bins
bin_start = (intervals // bin_width) * bin_width

histogram = (
    bin_start.value_counts()
    .sort_index()
    .rename_axis('bin_start')
    .reset_index(name='count')
)

histogram['bin_end'] = histogram['bin_start'] + bin_width

# Altair chart
chart = alt.Chart(histogram).mark_bar().encode(
    x=alt.X(
        'bin_start:Q',
        title='Time interval between different users (seconds)',
        axis=alt.Axis(tickMinStep=bin_width)
    ),
    x2='bin_end:Q',
    y=alt.Y('count:Q', title='Number of occurrences'),
    tooltip=[
        alt.Tooltip('bin_start:Q', title='From (seconds)'),
        alt.Tooltip('bin_end:Q', title='To (seconds)'),
        alt.Tooltip('count:Q', title='Occurrences')
    ]
).properties(
    title='Distribution of intervals between consecutive events from different users',
    width=750,
    height=400
)

chart

239957 events


alt.Chart(...)

Sabemos que:
- Entre cliente y cliente suelen haber entre 0,04 segundos y 0,07 segundos.
    - Tampoco encaja con 3 por segundo, son entre 14 y 25 por segundo.
- Un mismo cliente hace muchas cosas, porque hay alrededor de 150 eventos por segundo.

In [ ]:
# sql("SELECT * FROM shop.events ORDER BY ts DESC LIMIT 5").head()

,device,event_type,price_eur,product_id,region,session_id,ts,user_id
0,web,view,78.99,p_7c601e,Andalucía,s_fdeab60848f2,2026-11-14 14:59:30.977000000,u_7382e663
1,web,view,31.99,p_0ceafe,Andalucía,s_fdeab60848f2,2026-11-14 14:59:30.970000000,u_7382e663
2,web,view,180.99,p_84ce06,Andalucía,s_fdeab60848f2,2026-11-14 14:59:30.963000000,u_7382e663
3,web,view,216.99,p_1f2b64,Andalucía,s_fdeab60848f2,2026-11-14 14:59:30.956000000,u_7382e663
4,web,view,95.99,p_44e05b,Andalucía,s_fdeab60848f2,2026-11-14 14:59:30.948000000,u_7382e663


- ¿Cada cuánto se actualiza un producto?

In [110]:
actualizaciones_df = sql("SELECT ts FROM shop.catalog_updates ORDER BY ts")
actualizaciones_df.head(2)

,ts
0,2026-11-14 09:00:00.000000000
1,2026-11-14 09:00:00.000000000


In [111]:
df = actualizaciones_df.copy()

# Convert strings to datetime
df['ts'] = pd.to_datetime(df['ts'])

# Calculate time difference between consecutive rows (in minutes)
df['intervalo_min'] = df['ts'].diff().dt.total_seconds() / 60

# Boxplot
chart = alt.Chart(df.dropna()).mark_boxplot(
    extent='min-max'
).encode(
    y=alt.Y(
        'intervalo_min:Q',
        title='Time interval (minutes)'
    ),
    tooltip=[
        alt.Tooltip('intervalo_min:Q', title='Interval (min)')
    ]
).properties(
    title='Time intervals between consecutive rows',
    width=300,
    height=400
)

chart

alt.Chart(...)

Básicamente se actualizan cada 5 minutos.

In [112]:
df = actualizaciones_df.copy()

# Convert strings to datetime
df['ts'] = pd.to_datetime(df['ts'])

# Calculate time difference between consecutive rows (in minutes)
df['intervalo_min'] = df['ts'].diff().dt.total_seconds() / 60

# Exclude NaN and zero values
df_plot = df[df['intervalo_min'] > 0]

# Boxplot
chart = alt.Chart(df_plot).mark_boxplot(
    extent='min-max'
).encode(
    y=alt.Y(
        'intervalo_min:Q',
        title='Time interval (minutes)'
    ),
    tooltip=[
        alt.Tooltip('intervalo_min:Q', title='Interval (min)')
    ]
).properties(
    title='Time intervals between consecutive rows',
    width=300,
    height=400
)

chart

alt.Chart(...)

- ¿Cómo se relacionan marcas y categorías? ¿Las marcas tienen categorías predefinidas?

- ¿Cuántos usuarios hay?

In [115]:
usuarios = sql("SELECT DISTINCT user_id FROM shop.history_sessions")
usuarios.info()

<class 'pandas.DataFrame'>
RangeIndex: 59698 entries, 0 to 59697
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   user_id  59698 non-null  str  
dtypes: str(1)
memory usage: 466.5 KB


In [122]:
categorias = sql("SELECT COUNT (DISTINCT category) FROM shop.catalog")
categorias

,COUNTDistinct(category)
0,10


In [123]:
subcategorias = sql("SELECT COUNT (DISTINCT subcategory) FROM shop.catalog")
subcategorias

,COUNTDistinct(subcategory)
0,89


In [124]:
marcas = sql("SELECT COUNT (DISTINCT brand) FROM shop.catalog")
marcas

,COUNTDistinct(brand)
0,80


90 according to README

In [126]:
devices = sql("SELECT COUNT (DISTINCT context.device) FROM shop.history_sessions")
devices

,COUNTDistinct(context.device)
0,3


In [125]:
devices = sql("SELECT COUNT (DISTINCT device) FROM shop.history_events")
devices

,COUNTDistinct(device)
0,3


In [127]:
regiones = sql("SELECT COUNT (DISTINCT context.region) FROM shop.history_sessions")
regiones

,COUNTDistinct(context.region)
0,10


In [128]:
regiones = sql("SELECT COUNT (DISTINCT region) FROM shop.history_events")
regiones

,COUNTDistinct(region)
0,10


In [129]:
paginas = sql("SELECT COUNT (DISTINCT context.page) FROM shop.history_sessions")
paginas

,COUNTDistinct(context.page)
0,3
